# Logistic regression in clinical and public health research

**Authors:** Renato Carneiro de Freitas Chaves, Tiago Mendonça dos Santos, Thiago Domingos Corrêa  

## 1) Purpose 

This notebook provides a clear, step-by-step guide to logistic regression in clinical and public health research.
The applied objective is to evaluate whether selected clinical characteristics are associated with hospital mortality, a binary outcome coded as 0 for survival and 1 for death.

The analysis addresses four practical questions:
1. How should variables, coding, missingness, and event frequency be inspected before fitting a logistic model?
2. How can unadjusted and adjusted logistic regression models be fitted in Python?
3. How should odds ratios, 95% confidence intervals, and predicted probabilities be reported?
4. How can discrimination and calibration be assessed in a simple educational format?

## 2) Required Libraries

This notebook uses standard scientific Python libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.metrics import roc_auc_score

pd.set_option("display.precision", 4)


## 3) Data Import (Excel)

This guide uses the Excel file **`data.xlsx`** with a sheet named **`Data`**.

- If `data.xlsx` is in the same folder as this notebook, you can keep the path as `"data.xlsx"`.
- Otherwise, replace with the full path to your file.

In [ ]:
# Update the path if needed
excel_path = "data.xlsx"
sheet_name = "Data"

data = pd.read_excel(excel_path, sheet_name=sheet_name)

# Quick inspection
data.head(), data.shape

## 4) Variable definitions and data inspection

Before conducting statistical analysis, the investigator should inspect the structure of the dataset, confirm the variable names, and review the first rows.

The main variables used in this report are:
- `patient`: patient identifier;
- `age`: patient age at ICU admission, in years;
- `creatinine`: serum creatinine level, in mg/dL;
- `saps_3`: Simplified Acute Physiology Score 3 at ICU admission;
- `chronic_kidney_disease`: chronic kidney disease indicator, coded as 0/1 or No/Yes;
- `outcome`: hospital mortality, coded as 0 = alive and 1 = dead.


In [ ]:
# Number of rows and columns.
print("Dataset dimensions:", data.shape)

# Variable names.
print("\nVariable names:")
print(list(data.columns))

# First six records.
data.head(6)

## 5) Basic data preparation

The main analysis uses complete observations for the outcome and selected predictors.

In [ ]:
required_variables = [
    "patient", "age", "saps_3", "creatinine",
    "chronic_kidney_disease", "outcome"
]

analysis_data = data[required_variables].copy()

# Convert continuous variables to numeric.
for variable in ["age", "saps_3", "creatinine"]:
    analysis_data[variable] = pd.to_numeric(analysis_data[variable], errors="coerce")

# Remove records with missing values in any variable used in the analyses.
analysis_data = analysis_data.dropna(subset=[
    "age", "saps_3", "creatinine", "chronic_kidney_disease", "outcome"
]).copy()


# Logistic regression requires both events and nonevents.
event_counts = analysis_data["outcome"].value_counts().sort_index()
print(event_counts)

if analysis_data["outcome"].nunique() < 2:
    raise ValueError("The outcome must contain both events and nonevents for logistic regression.")

analysis_data.head()


## 6) Descriptive clinical table

The descriptive table summarizes the analytic dataset before model fitting. This step is essential because it confirms the sample size, event frequency, and distribution of key predictors.


In [ ]:
def mean_sd(series, digits=1):
    return f"{series.mean():.{digits}f} ({series.std(ddof=1):.{digits}f})"


def median_iqr(series, digits=2):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    return f"{series.median():.{digits}f} ({q1:.{digits}f} to {q3:.{digits}f})"


def n_percent(condition, digits=1):
    n = int(condition.sum())
    percent = 100 * condition.mean()
    return f"{n} ({percent:.{digits}f}%)"


descriptive_table = pd.DataFrame({
    "Characteristic": [
        "Sample size",
        "Age, mean (SD), years",
        "SAPS 3, mean (SD)",
        "Creatinine, median (IQR), mg/dL",
        "Chronic kidney disease, n (%)",
        "Hospital mortality, n (%)"
    ],
    "Value": [
        str(len(analysis_data)),
        mean_sd(analysis_data["age"]),
        mean_sd(analysis_data["saps_3"]),
        median_iqr(analysis_data["creatinine"]),
        n_percent(analysis_data["chronic_kidney_disease"] == "Yes"),
        n_percent(analysis_data["outcome"] == 1)
    ]
})

display(descriptive_table)


## 7) Exploratory graph

A binary outcome can be plotted against a continuous predictor, but the observed values can only be 0 or 1. The fitted curve below shows the estimated probability of death across age from a univariable logistic model.

In [ ]:
age_only_model = smf.logit("outcome ~ age", data=analysis_data).fit(disp=False)

age_grid = pd.DataFrame({
    "age": np.linspace(analysis_data["age"].min(), analysis_data["age"].max(), 100)
})
age_grid["predicted_probability"] = age_only_model.predict(age_grid)

plt.figure(figsize=(7, 5))
plt.scatter(analysis_data["age"], analysis_data["outcome"])
plt.plot(age_grid["age"], age_grid["predicted_probability"], linewidth=2)
plt.xlabel("Age at admission (years)")
plt.ylabel("Hospital mortality (0 = alive, 1 = dead)")
plt.title("Observed binary outcome and fitted logistic curve")
plt.show()


## 8) Unadjusted logistic regression

The unadjusted model evaluates the association between age and hospital mortality without adjustment for other predictors.
Interpretation: the age odds ratio is the multiplicative change in the odds of death for each additional year of age, before adjustment for severity of illness, kidney function, or chronic kidney disease.


In [ ]:
def logistic_results_table(model):
    """Return coefficients, standard errors, odds ratios, 95% CIs, and P values."""
    params = model.params
    se = model.bse
    ci = model.conf_int(alpha=0.05)

    table = pd.DataFrame({
        "Term": params.index,
        "Coefficient_log_odds": params.values,
        "Standard_error": se.values,
        "OR": np.exp(params.values),
        "CI_low": np.exp(ci[0].values),
        "CI_high": np.exp(ci[1].values),
        "P_value": model.pvalues.values
    })
    return table


unadjusted_model = smf.logit("outcome ~ age", data=analysis_data).fit(disp=False)
unadjusted_results = logistic_results_table(unadjusted_model)

display(unadjusted_results.round(4))


## 9) Adjusted logistic regression

The adjusted model estimates the association between each predictor and mortality conditional on the other variables in the model.
Interpretation: an adjusted odds ratio is conditional on all other variables in the model. It should not be interpreted as a risk ratio or as a direct multiplicative change in probability.


In [ ]:
adjusted_formula = "outcome ~ age + saps_3 + creatinine + C(chronic_kidney_disease)"
adjusted_model = smf.logit(adjusted_formula, data=analysis_data).fit(disp=False)
adjusted_results = logistic_results_table(adjusted_model)

display(adjusted_results.round(4))


## 10) Rescaling continuous predictors

For continuous predictors, the unit of measurement determines the interpretation.
A one-year age odds ratio may be numerically small. A 10-year age odds ratio is often more clinically interpretable.

In [ ]:
age_beta = adjusted_model.params["age"]
age_se = adjusted_model.bse["age"]

age_10_year_table = pd.DataFrame({
    "Measure": ["Adjusted odds ratio for age per 10-year increase"],
    "OR": [np.exp(10 * age_beta)],
    "CI_low": [np.exp(10 * (age_beta - 1.96 * age_se))],
    "CI_high": [np.exp(10 * (age_beta + 1.96 * age_se))]
})

display(age_10_year_table.round(3))


## 11) Predicted probabilities

Logistic regression estimates coefficients on the log-odds scale. Predicted probabilities are often more intuitive for clinical communication.
Interpretation: predicted probabilities translate the fitted model into estimated absolute risks for clinically recognizable patient profiles. These probabilities should be interpreted cautiously if the model has not been externally validated.


In [ ]:
new_patients = pd.DataFrame({
    "age": [45, 65, 80],
    "saps_3": [35, 55, 75],
    "creatinine": [0.9, 1.3, 2.0],
    "chronic_kidney_disease": pd.Categorical(["No", "No", "Yes"], categories=["No", "Yes"])
})

new_patients["predicted_probability"] = adjusted_model.predict(new_patients)

display(new_patients.round(3))


## 12) Odds ratios, risk ratios, and event frequency

When the outcome is common, odds ratios move farther away from 1 than risk ratios.
Therefore, odds ratios from logistic regression should not be described as risk ratios or probability ratios.

Interpretation: when outcomes are common, the odds ratio is farther from 1 than the risk ratio. Therefore, odds ratios from logistic regression should not be described as risk ratios or probability ratios.

In [ ]:
comparison_table = pd.DataFrame({
    "Scenario": ["Uncommon outcome", "Common outcome"],
    "Risk_unexposed": [0.05, 0.30],
    "Risk_exposed": [0.10, 0.60]
})

comparison_table["Risk_ratio"] = (
    comparison_table["Risk_exposed"] / comparison_table["Risk_unexposed"]
)
comparison_table["Odds_ratio"] = (
    (comparison_table["Risk_exposed"] / (1 - comparison_table["Risk_exposed"])) /
    (comparison_table["Risk_unexposed"] / (1 - comparison_table["Risk_unexposed"]))
)

display(comparison_table.round(3))


## 13) Model performance: discrimination

Discrimination evaluates whether patients with events tend to receive higher predicted probabilities than patients without events. The C statistic is equivalent to the area under the receiver operating characteristic curve for binary outcomes.

Interpretation: a C statistic of 0.5 indicates no better ranking than chance, and a C statistic of 1.0 indicates perfect ranking. Discrimination does not assess whether predicted probabilities are well calibrated.


In [ ]:
analysis_data["predicted_probability"] = adjusted_model.predict(analysis_data)

ranked_predictions = analysis_data["predicted_probability"].rank(method="average")
n_event = int((analysis_data["outcome"] == 1).sum())
n_nonevent = int((analysis_data["outcome"] == 0).sum())
sum_ranks_event = ranked_predictions[analysis_data["outcome"] == 1].sum()

c_statistic_manual = (
    sum_ranks_event - n_event * (n_event + 1) / 2
) / (n_event * n_nonevent)

c_statistic_sklearn = roc_auc_score(
    analysis_data["outcome"], analysis_data["predicted_probability"]
)

c_statistic = c_statistic_manual

c_statistic_table = pd.DataFrame({
    "Measure": ["C statistic / AUROC", "AUROC check using sklearn"],
    "Value": [c_statistic_manual, c_statistic_sklearn]
})

display(c_statistic_table.round(3))


## 14) Model performance: calibration

Calibration evaluates whether predicted probabilities agree with observed event frequencies.
Grouped calibration can be summarized by comparing the mean predicted probability with the observed event frequency within ordered risk groups.

Interpretation: points close to the 45-degree line suggest agreement between predicted and observed risks. Grouped calibration plots are useful for teaching but do not replace internal validation, external validation, or more detailed calibration assessment in a formal prediction-model study.


In [ ]:
# Rank-based quintiles are robust to duplicated predicted probabilities.
analysis_data["risk_group"] = pd.qcut(
    analysis_data["predicted_probability"].rank(method="first"),
    q=5,
    labels=[f"Quintile {i}" for i in range(1, 6)]
)

calibration_table = (
    analysis_data
    .groupby("risk_group", observed=True)
    .agg(
        observed=("outcome", "mean"),
        predicted=("predicted_probability", "mean"),
        n=("outcome", "size")
    )
    .reset_index()
)

display(calibration_table.round(3))


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(calibration_table["predicted"], calibration_table["observed"])
plt.plot([0, 1], [0, 1], linewidth=2)
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed event frequency")
plt.title("Calibration plot by quintile of predicted risk")
plt.show()


## 15) Final summary table

The final table summarizes the main regression results and the discrimination of the adjusted model.

A simple formatting function is used for P values. This avoids unnecessary complexity while producing a report-ready table.


In [ ]:
def format_p(p):
    if pd.isna(p):
        return "Not applicable"
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def get_row(results_table, term):
    row = results_table.loc[results_table["Term"] == term]
    if row.empty:
        return None
    return row.iloc[0]


def format_or(row):
    if row is None:
        return "Not available"
    return f"OR {row['OR']:.3f}"


def format_ci(row):
    if row is None:
        return "Not available"
    return f"95% CI {row['CI_low']:.3f} to {row['CI_high']:.3f}"


unadj_age = get_row(unadjusted_results, "age")
adj_age = get_row(adjusted_results, "age")
adj_saps = get_row(adjusted_results, "saps_3")

final_summary_table = pd.DataFrame({
    "Analysis": [
        "Unadjusted logistic regression: age",
        "Adjusted logistic regression: age",
        "Adjusted logistic regression: SAPS 3",
        "Adjusted model C statistic"
    ],
    "Estimate": [
        format_or(unadj_age),
        format_or(adj_age),
        format_or(adj_saps),
        f"{c_statistic:.3f}"
    ],
    "Confidence_interval": [
        format_ci(unadj_age),
        format_ci(adj_age),
        format_ci(adj_saps),
        "Not applicable"
    ],
    "P_value": [
        format_p(unadj_age["P_value"] if unadj_age is not None else np.nan),
        format_p(adj_age["P_value"] if adj_age is not None else np.nan),
        format_p(adj_saps["P_value"] if adj_saps is not None else np.nan),
        "Not applicable"
    ]
})

display(final_summary_table)
